# SEPA Entry Points: Identify and Evaluate

This notebook identifies and evaluates **Specific Entry Point Analysis (SEPA)** entry points using Mark Minervini's methodology. For each stock in a configurable universe, it:

1. **Detects base patterns**: Cup & Handle (Handle, 3C/Cup Completion Cheat, Low-Cheat), Double Bottom, Darvas Box, and Power Play.
2. **Identifies VCP contractions** (Volatility Contraction Pattern) and **pivot points** within these bases.
3. **Qualifies entries** using:
   - **Trend**: Uptrend status (50 > 150 > 200 SMA, 200d rising) and Trend Template score (0–8).
   - **Fab 5 (MPA)**: EPS growth (quarterly & annual), sales growth, ROE, and Relative Strength.
   - **Base/Pivot quality**: Base depth, duration, VCP characteristics, pivot tightness, and volume behavior.

The output is a ranked table of actionable entry points with all qualifiers, plus per-symbol deep dives.

## 1. Configuration

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.2f}".format)

ROOT = Path.cwd() if (Path.cwd() / "turtle_rpm").is_dir() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from turtle_rpm.sepa import (
    MIN_DAILY_BARS,
    compute_smas,
    find_bases,
    get_daily_ohlcv,
    pivot_forming,
    pivot_in_base,
    pivot_volume_vs_average,
    to_weekly,
    uptrend_at_date,
)
from turtle_rpm.leadership import (
    add_52w_high_low,
    rs_ratio_6m,
    trend_template,
)
from turtle_rpm.pivot_scan import pivot_quality_score

import yfinance as yf

print(f"Project root: {ROOT}")

In [ ]:
# ---------------------------------------------------------------------------
# Universe: edit this list with the symbols you want to evaluate.
# Use a focused watchlist for faster results; expand for broader screening.
# ---------------------------------------------------------------------------
SYMBOLS = [
    "AAPL", "MSFT", "NVDA", "AMZN", "META",
    "GOOGL", "AVGO", "TSLA", "LLY", "COST",
]

YEARS_OF_DATA = 5
MIN_TREND_TEMPLATE_SCORE = 5  # minimum TT score for a "quality" entry
MAX_DISTANCE_PCT = 5.0        # max % below resistance to consider "near buy point"

## 2. Data Loading

Load daily and weekly OHLCV for each symbol, compute SMAs, and prepare the data for analysis.

In [ ]:
data_cache: dict[str, dict] = {}

for sym in SYMBOLS:
    print(f"Loading {sym}...", end=" ")
    df_daily = get_daily_ohlcv(sym, years=YEARS_OF_DATA)
    if df_daily.empty or len(df_daily) < MIN_DAILY_BARS:
        print(f"SKIP (insufficient data: {len(df_daily)} bars)")
        continue
    daily_smas = compute_smas(df_daily)
    daily_52w = add_52w_high_low(daily_smas)
    weekly = to_weekly(df_daily)
    data_cache[sym] = {
        "daily": df_daily,
        "daily_smas": daily_smas,
        "daily_52w": daily_52w,
        "weekly": weekly,
    }
    print(f"OK ({len(df_daily)} daily bars, {len(weekly)} weekly bars)")

print(f"\nLoaded {len(data_cache)}/{len(SYMBOLS)} symbols successfully.")

## 3. Trend Qualification

For each symbol, evaluate the **Trend Template** (Minervini's 8 criteria) and **uptrend status** at the current date. The Trend Template score (0–8) measures how well the stock's price structure aligns with a Stage 2 uptrend.

In [ ]:
trend_rows = []

for sym, cache in data_cache.items():
    daily_smas = cache["daily_smas"]
    daily_52w = cache["daily_52w"]
    last_date = daily_smas.index[-1]
    close = float(daily_smas["Close"].iloc[-1])

    in_uptrend = uptrend_at_date(daily_smas, last_date)

    rs = rs_ratio_6m(sym)
    cache["rs_ratio"] = rs

    tt = trend_template(daily_52w, rs_ratio=rs)
    cache["trend_template"] = tt

    sma50 = float(daily_smas["SMA_50"].iloc[-1]) if pd.notna(daily_smas["SMA_50"].iloc[-1]) else None
    sma150 = float(daily_smas["SMA_150"].iloc[-1]) if pd.notna(daily_smas["SMA_150"].iloc[-1]) else None
    sma200 = float(daily_smas["SMA_200"].iloc[-1]) if pd.notna(daily_smas["SMA_200"].iloc[-1]) else None

    trend_rows.append({
        "Symbol": sym,
        "Close": close,
        "Uptrend": in_uptrend,
        "TT Score": tt["score"],
        "RS Ratio (6m)": round(rs, 2) if rs else None,
        "RS vs SPY": "Outperforming" if rs and rs >= 1.0 else "Underperforming",
        "SMA 50": round(sma50, 2) if sma50 else None,
        "SMA 150": round(sma150, 2) if sma150 else None,
        "SMA 200": round(sma200, 2) if sma200 else None,
    })

trend_df = pd.DataFrame(trend_rows)
print("\n=== Trend Qualification ===")
display(trend_df.style.map(
    lambda v: "background-color: #2d5a27" if v is True else ("background-color: #5a2727" if v is False else ""),
    subset=["Uptrend"]
))

### Trend Template Detail

Breakdown of the 8 Trend Template criteria for each symbol.

In [ ]:
tt_detail_rows = []
for sym, cache in data_cache.items():
    tt = cache["trend_template"]
    for d in tt["details"]:
        tt_detail_rows.append({
            "Symbol": sym,
            "Criterion": d["name"],
            "Pass": d["pass"],
            "Detail": d.get("detail", ""),
        })

tt_detail_df = pd.DataFrame(tt_detail_rows)
tt_pivot = tt_detail_df.pivot(index="Criterion", columns="Symbol", values="Pass")
tt_pivot = tt_pivot[list(data_cache.keys())]  # preserve symbol order

def _pass_fail_style(v):
    if v is True:
        return "background-color: #2d5a27; color: white"
    if v is False:
        return "background-color: #5a2727; color: white"
    return ""

print("\n=== Trend Template: 8 Criteria (True = Pass) ===")
display(tt_pivot.style.map(_pass_fail_style))

## 4. Base Detection

Detect all SEPA bases for each symbol. Bases include:
- **Cup & Handle** (3C/Cup Completion Cheat, Low-Cheat, Cup w/ Handle)
- **Double Bottom**
- **Darvas Box**
- **Power Play**

Each base is evaluated for depth, duration, resistance level, and distance to buy point.

In [ ]:
all_bases_rows = []

for sym, cache in data_cache.items():
    daily_smas = cache["daily_smas"]
    weekly = cache["weekly"]
    df_daily = cache["daily"]

    pivot = pivot_forming(df_daily)
    cache["pivot"] = pivot

    bases = find_bases(weekly, daily_smas, pivot=pivot)
    cache["bases"] = bases

    base_containing = pivot_in_base(pivot, bases) if pivot.get("forming") else None
    cache["base_containing_pivot"] = base_containing

    for b in bases:
        is_pivot_base = (base_containing is not None
                         and b["start_date"] == base_containing["start_date"]
                         and b["base_type"] == base_containing["base_type"])
        all_bases_rows.append({
            "Symbol": sym,
            "Base Type": b["base_type"],
            "Current": b.get("is_current", False),
            "Start": b["start_date"].strftime("%Y-%m-%d") if hasattr(b["start_date"], "strftime") else str(b["start_date"]),
            "End": b["end_date"].strftime("%Y-%m-%d") if hasattr(b["end_date"], "strftime") else str(b["end_date"]),
            "Depth %": b["depth_pct"],
            "Duration (wks)": b["duration_weeks"],
            "Resistance": b.get("resistance"),
            "Distance %": b.get("distance_pct"),
            "Buy Point Date": b.get("buy_point_date"),
            "VCP-like": b.get("vcp_like", False),
            "Pivot In Base": is_pivot_base,
        })

bases_df = pd.DataFrame(all_bases_rows)
if not bases_df.empty:
    bases_df = bases_df.sort_values(["Current", "Symbol"], ascending=[False, True])
    print(f"\n=== All Detected Bases ({len(bases_df)} total) ===")
    display(bases_df.reset_index(drop=True))
else:
    print("No bases detected for any symbol in the universe.")

### Base Type Distribution

Breakdown of detected bases by type across the universe.

In [ ]:
if not bases_df.empty:
    type_counts = bases_df["Base Type"].value_counts().reset_index()
    type_counts.columns = ["Base Type", "Count"]

    current_by_type = bases_df[bases_df["Current"]].groupby("Base Type").size().reset_index(name="Current Bases")
    type_summary = type_counts.merge(current_by_type, on="Base Type", how="left").fillna(0)
    type_summary["Current Bases"] = type_summary["Current Bases"].astype(int)

    avg_depth = bases_df.groupby("Base Type")["Depth %"].mean().reset_index(name="Avg Depth %")
    avg_dur = bases_df.groupby("Base Type")["Duration (wks)"].mean().reset_index(name="Avg Duration (wks)")
    type_summary = type_summary.merge(avg_depth, on="Base Type").merge(avg_dur, on="Base Type")

    print("\n=== Base Type Distribution ===")
    display(type_summary)
else:
    print("No bases to summarize.")

## 5. Pivot Detection & VCP Analysis

Identify **pivots** (tight consolidations of 3–10 days, < 8% range) within bases. Evaluate **VCP** (Volatility Contraction Pattern) characteristics: decreasing pullback depths, volume dry-up, and ATR contraction.

In [ ]:
pivot_rows = []

for sym, cache in data_cache.items():
    df_daily = cache["daily"]
    pivot = cache["pivot"]
    bases = cache["bases"]
    base_containing = cache["base_containing_pivot"]
    current_base = next((b for b in bases if b.get("is_current")), None)

    vol_label = pivot_volume_vs_average(df_daily, pivot) if pivot.get("forming") else None
    cache["pivot_volume"] = vol_label

    quality_row = {
        "pivot_forming": pivot.get("forming", False),
        "pivot_range_pct": pivot.get("range_pct"),
        "tight_closes": pivot.get("tight_closes", False),
        "in_base": base_containing is not None,
        "volume_at_pivot": vol_label,
        "distance_pct": base_containing.get("distance_pct") if base_containing else None,
        "buy_point_date": base_containing.get("buy_point_date") if base_containing else None,
    }
    quality = pivot_quality_score(quality_row)
    cache["pivot_quality"] = quality

    pivot_rows.append({
        "Symbol": sym,
        "Pivot Forming": pivot.get("forming", False),
        "Pivot Days": pivot.get("days"),
        "Range %": pivot.get("range_pct"),
        "Tight Closes": pivot.get("tight_closes", False),
        "Pivot High": pivot.get("pivot_high"),
        "In Base": base_containing is not None,
        "Base Type": base_containing.get("base_type") if base_containing else None,
        "Volume vs Avg": vol_label if vol_label else "N/A",
        "Pivot Quality": quality,
        "VCP-like": current_base.get("vcp_like", False) if current_base else False,
    })

pivot_df = pd.DataFrame(pivot_rows)
print("\n=== Pivot Detection & VCP Analysis ===")
display(pivot_df.sort_values("Pivot Quality", ascending=False).reset_index(drop=True))

### VCP Contraction Detail

For symbols with a current base, examine the VCP contraction characteristics in detail: pullback depths from high, volume behavior on up/down weeks, and ATR contraction.

In [ ]:
vcp_detail_rows = []

for sym, cache in data_cache.items():
    bases = cache["bases"]
    current_base = next((b for b in bases if b.get("is_current")), None)
    if current_base is None:
        continue

    weekly = cache["weekly"]
    df_daily = cache["daily"]
    start_ts = pd.Timestamp(current_base["start_date"])
    end_ts = pd.Timestamp(current_base["end_date"])
    prior_high = current_base["prior_high"]
    base_low = current_base["base_low"]

    # Pullback depths from prior high using weekly pivot lows in the base
    base_weekly = weekly[(weekly.index >= start_ts) & (weekly.index <= end_ts)]
    if base_weekly.empty or prior_high <= 0:
        continue

    # Find local lows (simple: weeks where Low < both neighbors)
    pullback_depths = []
    lows = base_weekly["Low"].values
    for i in range(1, len(lows) - 1):
        if lows[i] <= lows[i - 1] and lows[i] <= lows[i + 1]:
            depth = (prior_high - lows[i]) / prior_high * 100
            pullback_depths.append(round(depth, 2))
    if not pullback_depths and len(lows) > 0:
        depth = (prior_high - min(lows)) / prior_high * 100
        pullback_depths.append(round(depth, 2))

    contracting = len(pullback_depths) >= 2 and all(
        pullback_depths[i] > pullback_depths[i + 1] for i in range(len(pullback_depths) - 1)
    )

    # Volume: up-week vs down-week average
    if "Volume" in base_weekly.columns:
        up_weeks = base_weekly["Close"] >= base_weekly["Open"]
        up_vol = base_weekly.loc[up_weeks, "Volume"].mean()
        dn_vol = base_weekly.loc[~up_weeks, "Volume"].mean()
        vol_dryup = (not pd.isna(dn_vol) and not pd.isna(up_vol) and dn_vol < up_vol)
    else:
        up_vol = dn_vol = None
        vol_dryup = None

    # ATR contraction
    hl = df_daily["High"] - df_daily["Low"]
    hc = (df_daily["High"] - df_daily["Close"].shift(1)).abs()
    lc = (df_daily["Low"] - df_daily["Close"].shift(1)).abs()
    tr = pd.concat([hl, hc, lc], axis=1).max(axis=1)
    atr_series = tr.ewm(alpha=1.0 / 14, min_periods=14, adjust=False).mean()
    atr_in_base = atr_series[(atr_series.index >= start_ts) & (atr_series.index <= end_ts)].dropna()
    atr_contracting = None
    if len(atr_in_base) >= 2:
        atr_contracting = float(atr_in_base.iloc[-1]) < float(atr_in_base.iloc[0])

    vcp_detail_rows.append({
        "Symbol": sym,
        "Base Type": current_base["base_type"],
        "Pullback Depths %": " -> ".join(f"{d:.1f}" for d in pullback_depths),
        "Contracting": contracting,
        "# Contractions": len(pullback_depths),
        "Up-Wk Vol": f"{up_vol:,.0f}" if up_vol and not pd.isna(up_vol) else "N/A",
        "Dn-Wk Vol": f"{dn_vol:,.0f}" if dn_vol and not pd.isna(dn_vol) else "N/A",
        "Vol Dry-Up": vol_dryup,
        "ATR Contracting": atr_contracting,
        "VCP-like (combined)": current_base.get("vcp_like", False),
    })

if vcp_detail_rows:
    vcp_df = pd.DataFrame(vcp_detail_rows)
    print("\n=== VCP Contraction Detail (Current Bases Only) ===")
    display(vcp_df)
else:
    print("No current bases found for VCP analysis.")

## 6. Fab 5 Qualifiers (MPA)

Minervini's **Fab 5** fundamental/technical qualifiers from MPA (Minervini Private Access):

1. **Quarterly EPS Growth** >= 25% (current quarter vs same quarter prior year)
2. **Annual EPS Growth** >= 25% (year-over-year)
3. **Sales Growth** >= 25% (quarterly revenue growth)
4. **ROE** >= 17% (return on equity)
5. **Relative Strength** — outperforming SPY (RS ratio > 1.0)

Data sourced from yfinance where available.

In [ ]:
fab5_rows = []

for sym, cache in data_cache.items():
    ticker = yf.Ticker(sym)
    rs = cache.get("rs_ratio")

    # 1. Quarterly EPS growth (via quarterly income_stmt Net Income as proxy)
    eps_q_pass, eps_q_detail = False, "N/A"
    try:
        qi = ticker.quarterly_income_stmt
        if qi is not None and not qi.empty:
            ni_row = None
            for label in ["Net Income", "Net Income Common Stockholders"]:
                if label in qi.index:
                    ni_row = qi.loc[label]
                    break
            if ni_row is not None:
                ni = ni_row.dropna().sort_index(ascending=False)
                if len(ni) >= 5:
                    latest, prior = float(ni.iloc[0]), float(ni.iloc[4])
                elif len(ni) >= 2:
                    latest, prior = float(ni.iloc[0]), float(ni.iloc[1])
                else:
                    latest = prior = 0
                if prior != 0:
                    growth = (latest - prior) / abs(prior)
                    eps_q_pass = growth >= 0.25
                    eps_q_detail = f"{growth * 100:.1f}%"
    except Exception:
        pass

    # 2. Annual EPS growth (via income_stmt Net Income as proxy)
    eps_a_pass, eps_a_detail = False, "N/A"
    try:
        inc = ticker.income_stmt
        if inc is not None and not inc.empty:
            ni_row = None
            for label in ["Net Income", "Net Income Common Stockholders"]:
                if label in inc.index:
                    ni_row = inc.loc[label]
                    break
            if ni_row is not None:
                ni = ni_row.dropna().sort_index(ascending=False)
                if len(ni) >= 2:
                    latest, prior = float(ni.iloc[0]), float(ni.iloc[1])
                    if prior != 0:
                        growth = (latest - prior) / abs(prior)
                        eps_a_pass = growth >= 0.25
                        eps_a_detail = f"{growth * 100:.1f}%"
    except Exception:
        pass

    # 3. Sales growth (quarterly revenue)
    sales_pass, sales_detail = False, "N/A"
    try:
        qs = ticker.quarterly_financials
        if qs is not None and not qs.empty:
            rev_row = None
            for label in ["Total Revenue", "Revenue", "Operating Revenue"]:
                if label in qs.index:
                    rev_row = qs.loc[label]
                    break
            if rev_row is not None:
                rev = rev_row.dropna()
                if len(rev) >= 5:
                    latest_rev, prior_rev = float(rev.iloc[0]), float(rev.iloc[4])
                elif len(rev) >= 2:
                    latest_rev, prior_rev = float(rev.iloc[0]), float(rev.iloc[1])
                else:
                    latest_rev = prior_rev = 0
                if prior_rev > 0:
                    growth = (latest_rev - prior_rev) / prior_rev
                    sales_pass = growth >= 0.25
                    sales_detail = f"{growth * 100:.1f}%"
    except Exception:
        pass

    # 4. ROE >= 17%
    roe_pass, roe_detail = False, "N/A"
    try:
        info = ticker.info
        roe_val = info.get("returnOnEquity")
        if roe_val is not None:
            roe_pct = roe_val * 100 if abs(roe_val) < 5 else roe_val  # handle both decimal and pct
            roe_pass = roe_pct >= 17.0
            roe_detail = f"{roe_pct:.1f}%"
    except Exception:
        pass

    # 5. Relative Strength
    rs_pass = rs is not None and rs >= 1.0
    rs_detail = f"{rs:.2f}" if rs is not None else "N/A"

    fab5_score = sum([eps_q_pass, eps_a_pass, sales_pass, roe_pass, rs_pass])

    fab5_rows.append({
        "Symbol": sym,
        "Q EPS Growth": eps_q_detail,
        "Q EPS Pass": eps_q_pass,
        "A EPS Growth": eps_a_detail,
        "A EPS Pass": eps_a_pass,
        "Sales Growth": sales_detail,
        "Sales Pass": sales_pass,
        "ROE": roe_detail,
        "ROE Pass": roe_pass,
        "RS Ratio": rs_detail,
        "RS Pass": rs_pass,
        "Fab 5 Score": f"{fab5_score}/5",
    })

fab5_df = pd.DataFrame(fab5_rows)
print("\n=== Fab 5 Qualifiers (MPA) ===")

pass_cols = ["Q EPS Pass", "A EPS Pass", "Sales Pass", "ROE Pass", "RS Pass"]
display(fab5_df.style.map(
    _pass_fail_style,
    subset=pass_cols,
))

## 7. Entry Point Scoring & Ranking

Combine all qualifiers into a single **entry point score** and rank candidates. The score weights:

| Factor | Max Points | Description |
|--------|-----------|-------------|
| Trend Template | 24 | 3 pts per TT criterion passed (max 8 x 3 = 24) |
| Base Quality | 20 | Based on depth, duration, and type |
| Pivot Quality | 25 | Tightness, tight closes, in-base, volume, proximity |
| VCP | 10 | VCP-like contraction in current base |
| Fab 5 | 20 | 4 pts per fundamental qualifier passed |
| **Total** | **99** | |

Only symbols with a current base and a forming pivot are ranked as active entry points.

In [ ]:
def base_quality_score(base: dict | None) -> float:
    """Score a base 0–20 based on depth, duration, and type."""
    if base is None:
        return 0.0
    score = 0.0
    depth = base.get("depth_pct", 50)
    # Shallower bases are better (< 15% ideal, > 35% poor)
    if depth <= 15:
        score += 8
    elif depth <= 25:
        score += 6
    elif depth <= 35:
        score += 4
    else:
        score += 2

    dur = base.get("duration_weeks", 0)
    # Moderate duration is better (7–20 weeks ideal)
    if 7 <= dur <= 20:
        score += 6
    elif 4 <= dur <= 30:
        score += 4
    else:
        score += 2

    # Preferred base types
    btype = base.get("base_type", "")
    type_bonus = {
        "Cup w/ handle": 6,
        "Power Play": 6,
        "Double bottom": 5,
        "Cup completion cheat": 4,
        "Low cheat": 3,
        "Darvas box": 5,
    }
    score += type_bonus.get(btype, 2)
    return min(score, 20)


entry_rows = []

for sym, cache in data_cache.items():
    bases = cache["bases"]
    current_base = next((b for b in bases if b.get("is_current")), None)
    pivot = cache["pivot"]
    tt = cache["trend_template"]
    rs = cache.get("rs_ratio")

    # Trend Template score (0–24)
    tt_score = tt["score"] * 3

    # Base quality (0–20)
    bq_score = base_quality_score(current_base)

    # Pivot quality (0–25, from pivot_quality_score which is 0–100, rescale)
    pq_raw = cache.get("pivot_quality", 0)
    pq_score = min(pq_raw * 25 / 100, 25)

    # VCP (0–10)
    vcp_score = 10 if (current_base and current_base.get("vcp_like")) else 0

    # Fab 5 (0–20)
    fab5_row = next((r for r in fab5_rows if r["Symbol"] == sym), None)
    fab5_count = 0
    if fab5_row:
        fab5_count = sum([
            fab5_row.get("Q EPS Pass", False),
            fab5_row.get("A EPS Pass", False),
            fab5_row.get("Sales Pass", False),
            fab5_row.get("ROE Pass", False),
            fab5_row.get("RS Pass", False),
        ])
    fab5_score_val = fab5_count * 4

    total = tt_score + bq_score + pq_score + vcp_score + fab5_score_val

    has_entry = current_base is not None
    has_pivot = pivot.get("forming", False)
    distance = current_base.get("distance_pct") if current_base else None
    near_buy = distance is not None and distance <= MAX_DISTANCE_PCT

    if has_entry and distance is not None and current_base.get("buy_point_date") is not None:
        status = "At/Above Buy Point"
    elif has_entry and near_buy:
        status = f"Near Buy Point ({distance:.1f}% away)"
    elif has_entry:
        status = f"In Base ({distance:.1f}% away)" if distance else "In Base"
    else:
        status = "No Current Base"

    entry_rows.append({
        "Symbol": sym,
        "Status": status,
        "Base Type": current_base["base_type"] if current_base else None,
        "Depth %": current_base["depth_pct"] if current_base else None,
        "Duration": f"{current_base['duration_weeks']}w" if current_base else None,
        "Resistance": current_base.get("resistance") if current_base else None,
        "Distance %": distance,
        "Pivot": "Yes" if has_pivot else "No",
        "VCP": "Yes" if (current_base and current_base.get("vcp_like")) else "No",
        "TT": f"{tt['score']}/8",
        "Fab5": f"{fab5_count}/5",
        "Uptrend": cache.get("daily_smas") is not None and uptrend_at_date(cache["daily_smas"], cache["daily_smas"].index[-1]),
        "RS": f"{rs:.2f}" if rs else "N/A",
        "TT Pts": tt_score,
        "Base Pts": bq_score,
        "Pivot Pts": round(pq_score, 1),
        "VCP Pts": vcp_score,
        "Fab5 Pts": fab5_score_val,
        "Total Score": round(total, 1),
    })

entry_df = pd.DataFrame(entry_rows).sort_values("Total Score", ascending=False).reset_index(drop=True)

print("\n=== SEPA Entry Point Ranking ===")
print("Sorted by composite score (higher = more favorable entry).\n")
display(entry_df[[
    "Symbol", "Status", "Base Type", "Depth %", "Duration",
    "Resistance", "Distance %", "Pivot", "VCP", "TT", "Fab5",
    "Uptrend", "RS", "Total Score"
]])

### Score Breakdown

Detailed point breakdown by scoring category for each symbol.

In [ ]:
breakdown_df = entry_df[["Symbol", "TT Pts", "Base Pts", "Pivot Pts", "VCP Pts", "Fab5 Pts", "Total Score"]].copy()
breakdown_df.columns = ["Symbol", "Trend (24)", "Base (20)", "Pivot (25)", "VCP (10)", "Fab 5 (20)", "Total (99)"]

print("\n=== Score Breakdown ===")
display(breakdown_df.style.bar(subset=["Total (99)"], color="#3a7ca5", vmin=0, vmax=99))

## 8. Individual Symbol Deep Dive

For each symbol with a current base, display a comprehensive analysis including all bases, pivot details, trend template criteria, Fab 5, and VCP characteristics.

In [ ]:
from IPython.display import display, Markdown

for sym, cache in data_cache.items():
    bases = cache["bases"]
    pivot = cache["pivot"]
    tt = cache["trend_template"]
    rs = cache.get("rs_ratio")
    current_base = next((b for b in bases if b.get("is_current")), None)
    base_containing = cache.get("base_containing_pivot")

    display(Markdown(f"### {sym}"))

    # Price and trend summary
    daily_smas = cache["daily_smas"]
    close = float(daily_smas["Close"].iloc[-1])
    last_date = daily_smas.index[-1].strftime("%Y-%m-%d")
    in_uptrend = uptrend_at_date(daily_smas, daily_smas.index[-1])

    rs_str = f"{rs:.2f}" if rs else "N/A"
    display(Markdown(f"**Price:** ${close:.2f} (as of {last_date}) | "
                     f"**Uptrend:** {'Yes' if in_uptrend else 'No'} | "
                     f"**TT:** {tt['score']}/8 | "
                     f"**RS:** {rs_str}"))

    # Current base
    if current_base:
        bp_date = current_base.get('buy_point_date')
        bp_str = bp_date.strftime('%Y-%m-%d') if bp_date and hasattr(bp_date, 'strftime') else 'Not yet'
        display(Markdown(
            f"**Current Base:** {current_base['base_type']} | "
            f"Depth: {current_base['depth_pct']}% | "
            f"Duration: {current_base['duration_weeks']}w | "
            f"Resistance: ${current_base.get('resistance', 'N/A')} | "
            f"Distance: {current_base.get('distance_pct', 'N/A')}% | "
            f"Buy Point: {bp_str} | "
            f"VCP-like: {'Yes' if current_base.get('vcp_like') else 'No'}"
        ))
    else:
        display(Markdown("**Current Base:** None"))

    # Pivot
    if pivot.get("forming"):
        vol_label = cache.get("pivot_volume", "N/A")
        in_base_str = f"Yes ({base_containing['base_type']})" if base_containing else "No"
        display(Markdown(
            f"**Pivot:** {pivot['days']}d, {pivot['range_pct']}% range | "
            f"Tight closes: {'Yes' if pivot.get('tight_closes') else 'No'} | "
            f"High: ${pivot['pivot_high']} | "
            f"In base: {in_base_str} | "
            f"Volume: {vol_label}"
        ))
    else:
        display(Markdown("**Pivot:** Not forming"))

    # All bases table
    if bases:
        sym_bases = [{
            "Type": b["base_type"],
            "Current": b.get("is_current", False),
            "Start": b["start_date"].strftime("%Y-%m-%d") if hasattr(b["start_date"], "strftime") else str(b["start_date"]),
            "Depth %": b["depth_pct"],
            "Weeks": b["duration_weeks"],
            "Resistance": b.get("resistance"),
            "Dist %": b.get("distance_pct"),
            "VCP": b.get("vcp_like", False),
        } for b in bases]
        display(pd.DataFrame(sym_bases))
    else:
        display(Markdown("*No bases detected.*"))

    # Fab 5 for this symbol
    fab5_row = next((r for r in fab5_rows if r["Symbol"] == sym), None)
    if fab5_row:
        fab5_items = [
            f"Q EPS: {fab5_row['Q EPS Growth']} ({'Pass' if fab5_row['Q EPS Pass'] else 'Fail'})",
            f"A EPS: {fab5_row['A EPS Growth']} ({'Pass' if fab5_row['A EPS Pass'] else 'Fail'})",
            f"Sales: {fab5_row['Sales Growth']} ({'Pass' if fab5_row['Sales Pass'] else 'Fail'})",
            f"ROE: {fab5_row['ROE']} ({'Pass' if fab5_row['ROE Pass'] else 'Fail'})",
            f"RS: {fab5_row['RS Ratio']} ({'Pass' if fab5_row['RS Pass'] else 'Fail'})",
        ]
        display(Markdown("**Fab 5:** " + " | ".join(fab5_items)))

    display(Markdown("---"))

## 9. Actionable Entry Points Summary

Final filtered view: symbols that are **near or at a buy point**, with a qualifying trend and base. These are the most actionable SEPA entry points from the current universe.

In [ ]:
actionable = entry_df[
    (entry_df["Status"] != "No Current Base")
].copy()

if not actionable.empty:
    print(f"\n=== Actionable Entry Points ({len(actionable)} candidates) ===")
    print("Sorted by total score. Focus on symbols with high TT, Fab5, and pivot quality.\n")
    display(actionable[[
        "Symbol", "Status", "Base Type", "Depth %", "Duration",
        "Distance %", "Pivot", "VCP", "TT", "Fab5", "Uptrend", "Total Score"
    ]].reset_index(drop=True))

    # Highlight top pick
    top = actionable.iloc[0]
    print(f"\nTop candidate: {top['Symbol']} — {top['Status']}")
    print(f"  Base: {top['Base Type']}, Depth {top['Depth %']}%, {top['Duration']}")
    print(f"  Trend Template: {top['TT']}, Fab 5: {top['Fab5']}, VCP: {top['VCP']}")
    print(f"  Composite Score: {top['Total Score']}/99")
else:
    print("No actionable entry points found in the current universe.")
    print("Consider expanding the symbol list or checking back when bases form.")